# Model Training: Champion and Challenger Models

This notebook focuses on training two machine learning models — a Champion and a Challenger — for downstream evaluation and A/B testing.

Key steps in this notebook:
- Train the Champion model using Databricks AutoML.
- Train the Challenger model using Databricks AutoML.
- Register both models into Unity Catalog for sharing and deployment.


In [0]:
# 1. Import packages
import warnings
warnings.filterwarnings("ignore")

import databricks.automl
from mlflow import register_model
import mlflow

from datetime import datetime

# Set the MLflow registry URI to use Databricks Unity Catalog (UC) for model registry management
mlflow.set_registry_uri("databricks-uc")

(2025-04-22 20:28:00) WARNING: Hyperopt is deprecated for Databricks runtime for machine learning and will not be pre-installed in the next major version.


In [0]:
# Catalog name where the model artifacts will be stored in Unity Catalog
CATALOG_NAME = "alexander_booth" 
SCHEMA_NAME = "default" # Schema (database) name within the catalog to organize model artifacts
TABLE_NAME = "bike_sharing_training_data"
MODEL_NAME = "bike_sharing_uc_model"

## Machine Learning (Champion)

In [0]:
# 2. Define configs
input_table = f"{CATALOG_NAME}.{SCHEMA_NAME}.{TABLE_NAME}"
target_col = "cnt"
uc_model_name = f"{CATALOG_NAME}.{SCHEMA_NAME}.{MODEL_NAME}"

# Base experiment name
now = datetime.now().strftime("%Y%m%d_%H%M%S")  # Format: YYYYMMDD_HHMMSS
experiment_name = f"bike_sharing_uc_{now}"

# 3. Launch AutoML
summary = databricks.automl.regress(
    dataset = input_table,
    target_col = target_col,
    timeout_minutes = 5,
    primary_metric = "rmse",
    experiment_name = experiment_name
)

# Optional: print the best trial
# print(summary)

2025/04/22 20:28:50 INFO databricks.automl.client.manager: AutoML will optimize for root mean squared error metric, which is tracked as val_root_mean_squared_error in the MLflow experiment.
2025/04/22 20:28:51 INFO databricks.automl.client.manager: MLflow Experiment ID: 2818997009240085
2025/04/22 20:28:51 INFO databricks.automl.client.manager: MLflow Experiment: https://e2-demo-field-eng.cloud.databricks.com/?o=1444828305810485#mlflow/experiments/2818997009240085
2025/04/22 20:31:26 INFO databricks.automl.client.manager: Data exploration notebook: https://e2-demo-field-eng.cloud.databricks.com/?o=1444828305810485#notebook/2818997009240267
2025/04/22 20:34:48 INFO databricks.automl.client.manager: AutoML experiment completed successfully.


,Train,Validation,Test
root_mean_squared_error,49.310,52.424,50.594
mean_squared_error,2431.469,2748.301,2559.724
example_count,10441.000,3403.000,3535.000
r2_score,0.925,0.918,0.923
sum_on_target,1.971133e+06,651061.000,670485.000
score,0.925,0.918,0.923
mean_absolute_error,33.640,35.712,34.646
mean_on_target,188.788,191.320,189.670
max_error,425.622,361.907,346.710
mean_absolute_percentage_error,0.793,0.775,0.889


In [0]:
# 4. Best trial info
best_run_id = summary.best_trial.mlflow_run_id
best_model_uri = f"runs:/{best_run_id}/model"

print(f"Best Run ID: {best_run_id}")
print(f"Best Model URI: {best_model_uri}")

# 5. Register the best model into UC
registered_model = mlflow.register_model(
    model_uri = best_model_uri,
    name = uc_model_name
)

print(f"Registered Model Version: {registered_model.version}")

Best Run ID: 8bbd3cd166d54fab9ba1a71cf1c20bbb
Best Model URI: runs:/8bbd3cd166d54fab9ba1a71cf1c20bbb/model


Registered model 'alexander_booth.default.bike_sharing_uc_model' already exists. Creating a new version of this model...


Uploading artifacts:   0%|          | 0/11 [00:00<?, ?it/s]

Registered Model Version: 2


Created version '2' of model 'alexander_booth.default.bike_sharing_uc_model'.


In [0]:
# 6. Set model alias

# Initialize MLflow client
client = mlflow.MlflowClient()

# Set alias "prod" to the newly registered model version
client.set_registered_model_alias(
    name = uc_model_name,                       # Same model registered
    alias = "prod",
    version = registered_model.version          # Version registered
)

## Machine Learning (Challenger)

In [0]:
# 2. Define configs
input_table = f"{CATALOG_NAME}.{SCHEMA_NAME}.{TABLE_NAME}"
target_col = "cnt"
uc_model_name = f"{CATALOG_NAME}.{SCHEMA_NAME}.{MODEL_NAME}"

# Base experiment name
now = datetime.now().strftime("%Y%m%d_%H%M%S")  # Format: YYYYMMDD_HHMMSS
experiment_name = f"bike_sharing_uc_{now}"

# 3. Launch AutoML
summary = databricks.automl.regress(
    dataset = input_table,
    target_col = target_col,
    timeout_minutes = 10,
    primary_metric = "rmse",
    experiment_name = experiment_name
)

# Optional: print the best trial
# print(summary)

2025/04/22 20:36:48 INFO databricks.automl.client.manager: AutoML will optimize for root mean squared error metric, which is tracked as val_root_mean_squared_error in the MLflow experiment.
2025/04/22 20:36:49 INFO databricks.automl.client.manager: MLflow Experiment ID: 2818997009240453
2025/04/22 20:36:49 INFO databricks.automl.client.manager: MLflow Experiment: https://e2-demo-field-eng.cloud.databricks.com/?o=1444828305810485#mlflow/experiments/2818997009240453
2025/04/22 20:38:52 INFO databricks.automl.client.manager: Data exploration notebook: https://e2-demo-field-eng.cloud.databricks.com/?o=1444828305810485#notebook/2818997009240568
2025/04/22 20:47:28 INFO databricks.automl.client.manager: AutoML experiment completed successfully.


,Train,Validation,Test
root_mean_squared_error,38.350,45.524,43.308
mean_squared_error,1470.704,2072.428,1875.620
example_count,10441.000,3403.000,3535.000
r2_score,0.955,0.938,0.943
sum_on_target,1.971133e+06,651061.000,670485.000
score,0.955,0.938,0.943
mean_absolute_error,25.326,29.968,28.702
mean_on_target,188.788,191.320,189.670
max_error,398.603,367.256,395.017
mean_absolute_percentage_error,0.534,0.604,0.623


In [0]:
# 4. Best trial info
best_run_id = summary.best_trial.mlflow_run_id
best_model_uri = f"runs:/{best_run_id}/model"

print(f"Best Run ID: {best_run_id}")
print(f"Best Model URI: {best_model_uri}")

# 5. Register the best model into UC
registered_model = mlflow.register_model(
    model_uri = best_model_uri,
    name = uc_model_name
)

print(f"Registered Model Version: {registered_model.version}")

Best Run ID: b3dcc19d6a8949cd89b997839a036f12
Best Model URI: runs:/b3dcc19d6a8949cd89b997839a036f12/model


Registered model 'alexander_booth.default.bike_sharing_uc_model' already exists. Creating a new version of this model...


Uploading artifacts:   0%|          | 0/11 [00:00<?, ?it/s]

Registered Model Version: 3


Created version '3' of model 'alexander_booth.default.bike_sharing_uc_model'.


In [0]:
# 6. Set model alias

# Initialize MLflow client
client = mlflow.MlflowClient()

# Set alias "staging" to the newly registered model version
client.set_registered_model_alias(
    name = uc_model_name,                       # Same model registered
    alias = "staging",
    version = registered_model.version          # Version registered
)

## ✅ Next Steps

Now that the Champion and Challenger models are trained and registered, the next steps are:

1. **Add the models to a Delta Sharing share**  
   - Enable external consumers (or other workspaces) to access the models.
   - Share the model artifacts by adding them to an existing Delta Share.

2. **Accept the models in the production workspace**

3. **Set up model serving endpoints for A/B testing**

While Delta Sharing is available through the UI, we are presenting them programmatically to provide a complete, end-to-end example.

Let's move to the *02-delta-sharing-uc* notebook.